# Promptfoo 핵심 실습 튜토리얼

본 노트북은 **promptfoo 웹 UI 기반 레드팀 실행**에 필요한 최소 단계만 다룹니다.


## 실습 목표와 진행 흐름

### 실습 목표

1. promptfoo 설치에 필요한 런타임(Node.js, npm)만 확인합니다.
2. `promptfoo` CLI를 설치합니다.
3. `promptfoo redteam setup`으로 웹 UI를 열어 직접 설정하고 실행합니다.

### 진행 흐름

| 단계 | 내용 |
|------|------|
| 1 | Node.js / npm 확인 |
| 2 | promptfoo 설치 |
| 3 | `promptfoo redteam setup` 실행 |
| 4 | Promptfoo의 한계 |


## 사전 준비

- Node.js 20+ 및 npm
- 인터넷 연결 (npm 설치용)

> OpenAI 키는 웹 UI에서 직접 입력/설정합니다.


## 1) Node.js / npm 설치

다른 사용자 환경을 고려해 `node`/`npm`이 없으면 자동 설치를 시도합니다.

- macOS: Homebrew (`brew install node`)
- Linux: apt (`apt-get install -y nodejs npm`)
- Windows: winget (`winget install OpenJS.NodeJS.LTS`)


In [1]:
import platform, shutil, subprocess
from pathlib import Path

def run(cmd):
    print('▶', ' '.join(cmd)); subprocess.run(cmd, check=True)

node, npm = shutil.which('node'), shutil.which('npm')
if not (node and npm):
    os_name = platform.system().lower()
    if os_name == 'darwin':
        run([shutil.which('brew') or 'brew', 'install', 'node'])
    elif os_name == 'linux':
        run(['apt-get', 'update']); run(['apt-get', 'install', '-y', 'nodejs', 'npm'])
    elif os_name == 'windows':
        run(['winget', 'install', '--id', 'OpenJS.NodeJS.LTS', '-e'])
    else:
        raise RuntimeError(f'지원하지 않는 OS: {os_name}')

node, npm = shutil.which('node'), shutil.which('npm')
if not (node and npm):
    raise RuntimeError('node/npm 설치 후에도 PATH에서 찾을 수 없습니다. 터미널/커널 재시작 후 다시 실행하세요.')

print('node:', subprocess.run([node, '-v'], capture_output=True, text=True, check=True).stdout.strip())
print('npm :', subprocess.run([npm, '-v'], capture_output=True, text=True, check=True).stdout.strip())
NPM_BIN = npm
PROMPTFOO_DIR = Path.cwd() / 'promptfoo_report'; PROMPTFOO_DIR.mkdir(exist_ok=True)
print('workdir:', PROMPTFOO_DIR)


node: v24.14.1
npm : 11.11.0
workdir: /Users/selectstar/P5-1_Red-Teaming-Framework/promptfoo_report


## 2) promptfoo 설치


In [2]:
import shutil
import subprocess

subprocess.run([NPM_BIN, 'install', '-g', 'promptfoo'], check=True)
PROMPTFOO_BIN = shutil.which('promptfoo')
if not PROMPTFOO_BIN:
    raise RuntimeError('promptfoo 설치 후 PATH에서 찾을 수 없습니다. 터미널/커널을 재시작하세요.')

print('promptfoo:', subprocess.run([PROMPTFOO_BIN, '--version'], capture_output=True, text=True, check=True).stdout.strip())


npm warn ERESOLVE overriding peer dependency
npm warn While resolving: nunjucks@3.2.4
npm warn Found: chokidar@5.0.0
npm warn node_modules/promptfoo/node_modules/chokidar
npm warn   chokidar@"5.0.0" from promptfoo@0.121.12
npm warn   node_modules/promptfoo
npm warn     promptfoo@"*" from the root project
npm warn
npm warn Could not resolve dependency:
npm warn peerOptional chokidar@"^3.3.0" from nunjucks@3.2.4
npm warn node_modules/promptfoo/node_modules/nunjucks
npm warn   nunjucks@"^3.2.4" from promptfoo@0.121.12
npm warn   node_modules/promptfoo
npm warn
npm warn Conflicting peer dependency: chokidar@3.6.0
npm warn node_modules/chokidar
npm warn   peerOptional chokidar@"^3.3.0" from nunjucks@3.2.4
npm warn   node_modules/promptfoo/node_modules/nunjucks
npm warn     nunjucks@"^3.2.4" from promptfoo@0.121.12
npm warn     node_modules/promptfoo
npm warn EBADENGINE Unsupported engine {
npm warn EBADENGINE   package: 'mute-stream@4.0.0',
npm warn EBADENGINE   required: { node: '^22.22.2 


changed 713 packages in 32s

138 packages are looking for funding
  run `npm fund` for details
promptfoo: 0.121.12


## 3) `promptfoo redteam setup` 실행

아래 셀 실행 후 브라우저 웹 UI에서 다음을 설정합니다.

1. OpenAI API Key
2. Target model
3. Plugin / Strategy
4. Run 실행


In [4]:
import subprocess

cmd = [PROMPTFOO_BIN, 'redteam', 'setup', '--port', '15500']
print(' '.join(cmd))
subprocess.run(cmd, cwd=str(PROMPTFOO_DIR), check=True)


/Users/selectstar/.nvm/versions/node/v24.14.1/bin/promptfoo redteam setup --port 15500


(node:80560) ExperimentalWarning: DecompressInterceptor is experimental and subject to change
(Use `node --trace-warnings ...` to show where the warning was created)


Port 15500 is already in use. Do you have another Promptfoo instance running?


CalledProcessError: Command '['/Users/selectstar/.nvm/versions/node/v24.14.1/bin/promptfoo', 'redteam', 'setup', '--port', '15500']' returned non-zero exit status 1.

## 4) Promptfoo의 한계

### 1. 생성기 의존성

- `redteam` 공격 프롬프트는 생성 모델 품질에 영향을 받습니다.
- 생성기가 약하면 취약점 탐지율이 과소평가될 수 있습니다.

### 2. 플러그인 커버리지 한계

- 기본 plugin/strategy만으로는 도메인 특화 공격(금융/의료/내부업무)을 충분히 커버하기 어렵습니다.
- 실제 서비스 평가에는 커스텀 시나리오를 반드시 추가해야 합니다.

### 3. 결과 비결정성

- 공격 생성과 모델 응답 모두 확률적이라 재실행 시 결과가 달라질 수 있습니다.
- 단일 실행 결과만으로 안전성 결론을 내리기 어렵습니다.

### 4. 정량 지표의 한계

- pass/fail 또는 취약점 수치만으로 실제 위험도를 완전히 설명할 수 없습니다.
- 고위험 케이스는 응답 원문을 사람 검토로 확인해야 합니다.

### 5. 운영 맥락 미반영 가능성

- 단순 모델 호출 테스트는 실제 서비스의 시스템 프롬프트, 툴 호출, RAG, 권한 제어를 반영하지 못할 수 있습니다.
- 운영 환경 기준 검증은 실제 엔드포인트 연동 테스트가 필요합니다.
